# Week 3, day 4 (morning) — Worksheet 05 SOLUTIONS: source-to-target mapping and staging

Executed in the lab image. Every quoted number is what it actually printed.

Question 8 is the one worth arguing about. A third of the target model does not
exist anywhere in the source, and that is not a gap — it is the work.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — Source-to-target mapping and staging. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

SOURCE_TABLES = ["category", "city", "cohort", "course", "discount_type",
                 "employee", "employee_type", "enrollment", "payment_type",
                 "program", "students", "transaction"]

# The target model, exactly as slide 29 draws it.
TARGET = {
    "dim_program":  ["program_id", "source_program_id", "program_name",
                     "program_category", "is_active"],
    "dim_course":   ["course_id", "source_course_id", "course_name",
                     "credit_hours", "is_active"],
    "dim_cohort":   ["cohort_id", "source_cohort_id", "cohort_name",
                     "start_date", "end_date"],
    "dim_student":  ["student_id", "source_student_id", "student_name",
                     "date_of_birth", "city", "state", "country", "is_active"],
    "dim_date":     ["date_id", "full_date", "day_of_week", "day", "week",
                     "month", "quarter", "year", "is_weekend"],
    "dim_promotion": ["promotion_id", "source_discount_type_id",
                      "promotion_name", "discount_amount"],
    "fact_enrollment": ["enrollment_id", "program_id", "course_id", "cohort_id",
                        "student_id", "enrollment_date_id", "promotion_id",
                        "enrollment_count", "tuition_amount", "discount_amount",
                        "net_tuition_amount", "amount_paid_to_date",
                        "is_paid_in_full"],
}

print("source tables:", len(SOURCE_TABLES))
print("target tables:", len(TARGET), "->", ", ".join(TARGET))

PART A — the mapping

### Question 1

Write the source-to-target mapping for `dim_course` in slide 33's format: target column, source table/column, and the ETL logic. Print it as a table, then verify each named source column actually exists.

In [ ]:
MAPPING_DIM_COURSE = [
    ("course_id",        "(generated)",        "surrogate key, assigned at load"),
    ("source_course_id", "course.course_id",   "direct mapping; keeps the source key"),
    ("course_name",      "course.course_name", "direct mapping"),
    ("credit_hours",     "course.hours",       "rename hours -> credit_hours"),
    ("is_active",        "course.active_flg",  "cast 1/0 to boolean"),
]
print("%-18s %-22s %s" % ("TARGET COLUMN", "SOURCE", "ETL LOGIC"))
for t, s, logic in MAPPING_DIM_COURSE:
    print("%-18s %-22s %s" % (t, s, logic))
print()
crs = load("course")
for _, s, _ in MAPPING_DIM_COURSE:
    if "." in s:
        tbl, col = s.split(".")
        print("  %-22s exists: %s" % (s, col in load(tbl).columns))

```
TARGET COLUMN      SOURCE                 ETL LOGIC
course_id          (generated)            surrogate key, assigned at load
source_course_id   course.course_id       direct mapping; keeps the source key
course_name        course.course_name     direct mapping
credit_hours       course.hours           rename hours -> credit_hours
is_active          course.active_flg      cast 1/0 to boolean

  course.course_id       exists: True
  course.course_name     exists: True
  course.hours           exists: True
  course.active_flg      exists: True
```

Five target columns, and only two are pure copies. That ratio is normal, and it
is why the mapping document exists — the interesting rows are the ones that are
not "direct mapping".

**`course_id` has no source at all.** It is a surrogate key, generated at load
time, and question 9 shows why that is worth doing.

**`credit_hours` is a rename.** The source calls it `hours`, which is ambiguous —
hours of what? — and slide 29 calls the target `credit_hours`. This is the
cheapest kind of improvement a warehouse makes and one of the most valuable:
every analyst who reads `credit_hours` understands it, and nobody has to ask.

**`is_active` is a cast.** `active_flg` holds 1 and 0; the target is a boolean.
Trivial, and it is the kind of trivial that is worth writing down, because the
alternative is every downstream query guessing whether 1 means active.

The verification loop at the end is the point of doing this in a notebook rather
than a spreadsheet. A mapping is a set of claims about columns that exist, and
those claims rot: someone renames a source column and the document does not
change. Four lines of code check every claim in the mapping, and they can run in
CI.

Slide 41 lists what an ETL specification must document, and *"source tables and
columns used"* is the second item. A mapping that has never been executed against
the source is a document, not a specification.

### Question 2

Slide 34 says staging tables *keep data close to the original source structure* and *preserve source keys, timestamps, and source metadata*. Build `stg_enrollment` that way — every source column unchanged, plus `_loaded_at` and `_source_file`. Print its shape and columns.
> **NOTE:** resist the urge to clean anything here. Staging is a faithful copy; transformation is Step 3.

In [ ]:
enr = load("enrollment")
stg_enrollment = enr.copy()
stg_enrollment["_source_file"] = "enrollment.csv"
stg_enrollment["_loaded_at"] = pd.Timestamp("2026-08-29 09:00:00")

print("source enrollment: ", enr.shape)
print("stg_enrollment:    ", stg_enrollment.shape)
print()
print("columns:", list(stg_enrollment.columns))
print()
print("every source column preserved:",
      all(c in stg_enrollment.columns for c in enr.columns))
print("row count unchanged:", len(stg_enrollment) == len(enr))
print("values unchanged:   ",
      stg_enrollment[enr.columns].equals(enr))

```
source enrollment:  (2400, 6)
stg_enrollment:     (2400, 8)

columns: ['enrl_id', 'enrl_date', 'stu_id', 'course_id', 'cohort_id', 'status', '_source_file', '_loaded_at']

every source column preserved: True
row count unchanged: True
values unchanged:    True
```

Six source columns plus two metadata columns, and the three assertions confirm
what slide 34 asks for: same rows, same values, nothing cleaned.

The discipline here is **restraint**. Everything found so far is visible in this
table and none of it is fixed: `status` still says `cancelled` for 117 rows, the
TEST students are still here, the seven orphan `stu_id` values are untouched. That
is correct. Staging is a faithful copy; Step 3 is where rules get applied, and
worksheet 06 is Step 3.

The reason to separate them is the one from worksheet 02's `RAW` / `CORE` split:
when a business rule turns out to be wrong — and it will — you fix the rule and
rebuild from staging. If the cleaning happened during extraction, the only way
back is to ask the source system for yesterday's data, and operational systems
overwrite.

The two underscore-prefixed columns are the same idea as `BATCH_ID` and
`INSERTED_AT` in the Snowflake ingestion lab: `_source_file` says where a row came
from, `_loaded_at` says when it arrived. They make a bad load reversible, because
`DELETE FROM stg_enrollment WHERE _loaded_at = '...'` removes exactly one run.

Note `_loaded_at` is a fixed timestamp here rather than `now()`, so the notebook
produces identical output every run. In production it is the load's start time —
one value for the whole batch, not per row, so every row of one run is deletable
as a unit.

### Question 3

Slide 34 lists which source tables to stage and why. Build that table for this case study: for each of the six target tables, print the source tables it needs.

In [ ]:
NEEDS = {
    "dim_program":     ["program", "category"],
    "dim_course":      ["course"],
    "dim_cohort":      ["cohort"],
    "dim_student":     ["students", "city"],
    "dim_date":        ["enrollment"],
    "dim_promotion":   ["discount_type"],
    "fact_enrollment": ["enrollment", "transaction", "discount_type",
                        "course", "program"],
}
for target, sources in NEEDS.items():
    print("  %-17s <- %s" % (target, ", ".join(sources)))
print()
used = sorted({t for s in NEEDS.values() for t in s})
print("source tables staged:  %2d  %s" % (len(used), ", ".join(used)))
unused = [t for t in SOURCE_TABLES if t not in used]
print("source tables NOT used: %2d  %s" % (len(unused), ", ".join(unused)))

```
  dim_program       <- program, category
  dim_course        <- course
  dim_cohort        <- cohort
  dim_student       <- students, city
  dim_date          <- enrollment
  dim_promotion     <- discount_type
  fact_enrollment   <- enrollment, transaction, discount_type, course, program

source tables staged:   9  category, city, cohort, course, discount_type, enrollment, program, students, transaction
source tables NOT used:  3  employee, employee_type, payment_type
```

Nine of twelve tables are needed. This is slide 34's staging table, filled in.

Three details are worth pulling out.

**`dim_date` comes from `enrollment`.** There is no date table in the source —
there rarely is. The date dimension is *generated* from the range of dates the
facts actually contain, which worksheet 08 does. It is the most purely
constructed table in the model.

**`fact_enrollment` needs five source tables**, more than any dimension. That is
normal: the fact table is where everything converges, and it needs `course` and
`program` not for their attributes but to resolve `course_id` into a
`program_id`, because the source `enrollment` row does not carry one. Worksheet
01 question 10 is why.

**Three tables are unused**, and each is a different kind of unused:

- `employee` and `employee_type` describe staff. The enrollment process does not
  measure staff, so they are correctly out of scope. If someone later asks "which
  program manager has the highest enrollment growth", they come back in.
- `payment_type` is the interesting one. It is *unused for `fact_enrollment`* and
  it is exactly the dimension `fact_payment` would need — the galaxy schema from
  worksheet 04 question 6. It is not irrelevant; it is out of scope for **this
  fact table**, which is a different statement.

Writing that distinction down is the value of the exercise. "We do not need
`payment_type` because we are not modelling payments yet" is a decision. "We
never mapped `payment_type`" is an omission, and they look identical six months
later.

### Question 4

Slide 38 says *"load dimensions before facts so fact rows can reference them"*. Derive the load order from question 3's dependencies and print it as numbered waves.

In [ ]:
WAVE = {
    1: ["stg_* (all source tables staged)"],
    2: ["dim_program", "dim_course", "dim_cohort", "dim_student",
        "dim_date", "dim_promotion"],
    3: ["fact_enrollment"],
}
for n, tables in WAVE.items():
    print("wave %d:" % n)
    for t in tables:
        print("    %s" % t)
print()
print("wave 2 tables have no dependency on each other -> loadable in parallel")
print("wave 3 needs every key in wave 2 to exist first")

```
wave 1:
    stg_* (all source tables staged)
wave 2:
    dim_program
    dim_course
    dim_cohort
    dim_student
    dim_date
    dim_promotion
wave 3:
    fact_enrollment

wave 2 tables have no dependency on each other -> loadable in parallel
wave 3 needs every key in wave 2 to exist first
```

Three waves, and the dependency graph is genuinely this shallow — which is a
property of a star schema worth appreciating. Every dimension depends only on
staging; the fact depends on all the dimensions; nothing depends on a fact.

**Wave 2 is fully parallel.** Six dimension loads with no ordering between them,
because in a star no dimension references another. That is a real operational
benefit: the six can run as six concurrent tasks, and the wave finishes in the
time of the slowest one rather than the sum.

A snowflake schema does not have this property. `dim_course_sf` references
`dim_program_sf` references `dim_category_sf`, so wave 2 becomes three
sub-waves that must run in order. Worksheet 04 measured the snowflake's cost in
joins and storage; this is its cost in load orchestration, and it is the one
nobody mentions.

**Wave 3 must be last**, and slide 38 says why: *"loading dimensions before facts
so fact rows can reference them"*. A fact row carries surrogate keys, and a
surrogate key that does not exist yet cannot be looked up. Question 10 shows
precisely what that failure looks like.

Two things this ordering does not solve, both worth knowing now:

**Late-arriving dimensions.** If a fact row references a course that is not in
`dim_course` — because the source was extracted at a different moment — the
lookup fails at load time. The standard answer is an `Unknown` member: a row in
every dimension with key `-1`, so unmatched facts land there instead of being
dropped. Worksheet 09 uses it.

**Truncate-and-reload versus incremental.** These waves describe one run. If
dimensions are rebuilt each night while facts accumulate, surrogate keys must be
**stable** across runs, or last night's fact rows now point at the wrong course.
That is what `source_course_id` is for, and question 9 is about it.

PART B — Step 2: join and filter

### Question 5

Build the working dataset slide 35 describes: start from `stg_enrollment` and join course, program, cohort and student context. Print the row count after each join and confirm the grain never changes.
> **NOTE:** every one of these must be `how="left"` and must not add rows. Check both.

In [ ]:
enr = load("enrollment")
work = enr.copy()
print("start (stg_enrollment)      %5d rows" % len(work))

steps = [
    ("course",  load("course")[["course_id", "course_name", "hours",
                                "program_id", "active_flg"]], "course_id"),
    ("program", load("program")[["program_id", "program_name",
                                 "category_id"]], "program_id"),
    ("category", load("category")[["category_id", "category_name"]],
     "category_id"),
    ("cohort",  load("cohort")[["cohort_id", "cohort_name", "start_dt",
                                "end_dt"]], "cohort_id"),
    ("students", load("students")[["stu_id", "stu_name", "birthday",
                                   "city_id"]], "stu_id"),
]
for name, df, key in steps:
    before = len(work)
    work = work.merge(df, on=key, how="left")
    flag = "ok" if len(work) == before else "FAN-OUT"
    print("+ %-10s %5d -> %5d  %s" % (name, before, len(work), flag))
print()
print("grain preserved:", work.enrl_id.is_unique)

```
start (stg_enrollment)       2400 rows
+ course      2400 ->  2400  ok
+ program     2400 ->  2400  ok
+ category    2400 ->  2400  ok
+ cohort      2400 ->  2400  ok
+ students    2400 ->  2400  ok

grain preserved: True
```

Five joins, and the row count never moves. This is slide 35 — *"joining staged
tables using source keys... keeping the working dataset at the correct business
grain"* — done and, more importantly, **verified**.

The check printed after every merge is the habit worth stealing. Three lines of
code, and it catches the single most common ELT defect: a join to a dimension
whose key is not unique, which silently multiplies fact rows. Worksheet 04
question 7 is that failure at 101x; in real pipelines it is usually 1.004x, which
is far worse because it survives review.

Two design points in this code.

**Every join is `how="left"`.** The enrollment is the spine, and nothing is
allowed to remove one. If `course_id` 214 were missing from `course`, an inner
join would silently drop those enrollments; a left join keeps the row with nulls
in the course columns, where a data-quality check will find it. Worksheet 10 runs
that check.

**Only the needed columns are selected.** `load("course")[["course_id",
"course_name", "hours", "program_id", "active_flg"]]` rather than the whole
table. Two reasons: it avoids column-name collisions producing `_x` and `_y`
suffixes, and it makes the mapping from question 1 auditable — the columns in the
code are exactly the columns in the document.

`grain preserved: True` is `enrl_id.is_unique`, the same assertion worksheet 02
question 10 argued for. It is one line, it runs in a second on 2,400 rows and in
a few seconds on millions, and it is the assertion that turns "I think this join
was safe" into "this join was safe".

### Question 6

Slide 35 says *"filtering out test, duplicate, cancelled, or inactive records when required"*. Count each candidate for exclusion separately, and the total rows affected once overlaps are accounted for.
> **NOTE:** count them as sets, not by adding the four numbers up. A row can be more than one kind of problem.

In [ ]:
enr, stu = load("enrollment"), load("students")
test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
known_ids = set(stu.stu_id)

flags = {
    "test student":      set(enr.loc[enr.stu_id.isin(test_ids), "enrl_id"]),
    "cancelled":         set(enr.loc[enr.status == "cancelled", "enrl_id"]),
    "unknown student":   set(enr.loc[~enr.stu_id.isin(known_ids), "enrl_id"]),
}
for name, ids in flags.items():
    print("  %-18s %4d enrollments" % (name, len(ids)))
print()
union = set().union(*flags.values())
print("  sum of the three:  %4d" % sum(len(v) for v in flags.values()))
print("  distinct rows:     %4d" % len(union))
print("  overlap:           %4d" % (sum(len(v) for v in flags.values()) - len(union)))
print()
print("enrollments remaining if all are excluded:", len(enr) - len(union))

```
  test student         18 enrollments
  cancelled           117 enrollments
  unknown student       7 enrollments

  sum of the three:   142
  distinct rows:      141
  overlap:              1

enrollments remaining if all are excluded: 2259
```

Three exclusion candidates, and **the three counts do not add up**: 142 by
addition, 141 distinct. One enrollment is both cancelled and something else.

That single row is the reason the question insisted on sets. Adding up filter
counts is a habit that works right up until two filters overlap, and then the
report says 142 rows were excluded when 141 were. On this data the error is one
row; on a real system with eight overlapping rules it is routinely 10-20%, and it
always overstates, so the pipeline appears to be doing more cleaning than it is.

Each of the three is a genuinely different decision:

**Test students (18).** These should go. They are not business events; they are QA
artefacts, identifiable only because someone typed `TEST` in a name field. Note
how fragile that is — a test account named `Jane Smith` is undetectable. The real
fix is upstream (a flag on the account, a separate environment), and the
worksheet-level fix is to name the rule and count what it removes.

**Cancelled (117).** This is not a data-quality problem. It is a **business
question**, and slide 36 offers both answers — *"excluded or flagged"*. Finance
wants them out of revenue; admissions wants them counted, because cancellation
rate is a metric. Worksheet 06 takes a position.

**Unknown student (7).** A referential-integrity failure: the enrollment
references a `stu_id` that is not in `students`. Deleting these is the *worst*
option, because it hides a real problem — either the extract is inconsistent or
the source deleted a student who had enrollments. The standard handling is an
`Unknown` dimension member, so the rows stay countable and the anomaly stays
visible. Worksheet 09 does that.

**2,259 of 2,400** survive all three — a 5.9% reduction. Whether that is right
depends entirely on the cancelled decision, which is 117 of the 141.

PART C — auditing the mapping

### Question 7

Mapping completeness, first direction. Across the twelve source tables, count the total columns and list the ones no target column uses.
> **NOTE:** an unused column is a question, not a defect. Some are genuinely irrelevant; some mean a requirement was missed.

In [ ]:
USED = {
    "course": ["course_id", "course_name", "hours", "active_flg", "program_id"],
    "program": ["program_id", "program_name", "category_id", "active_flg"],
    "category": ["category_id", "category_name"],
    "cohort": ["cohort_id", "cohort_name", "start_dt", "end_dt"],
    "students": ["stu_id", "stu_name", "birthday", "city_id", "active_flg"],
    "city": ["city_id", "city_name", "provn_name", "cntry_name"],
    "enrollment": ["enrl_id", "enrl_date", "stu_id", "course_id",
                   "cohort_id", "status"],
    "transaction": ["enrl_id", "full_price", "discount_type_id",
                    "payment_amount", "full_paid"],
    "discount_type": ["discount_type_id", "discount_type_name",
                      "discount_amount"],
}
total = unused_n = 0
for name in SOURCE_TABLES:
    cols = list(load(name).columns)
    total += len(cols)
    unused = [c for c in cols if c not in USED.get(name, [])]
    unused_n += len(unused)
    if unused:
        print("  %-14s %s" % (name, ", ".join(unused)))
print()
print("source columns total:  %3d" % total)
print("used by the model:     %3d" % (total - unused_n))
print("unused:                %3d  (%.0f%%)" % (unused_n, 100 * unused_n / total))

```
  category       category_desc, director_id, start_date, active_flg
  city           provn_id, cntry_id
  course         course_desc, gov_code, full_time
  discount_type  discount_id
  employee       emp_id, emp_name, emp_type_id, status, city_id, title, manager_id, active_flg
  employee_type  emp_type_id, emp_type_des
  payment_type   pymt_type_id, pymt_type_name
  program        program_desc, pm_id, start_date
  students       emp_type_id, title, home_address, postal_code, phone_num, email

source columns total:   72
used by the model:      38
unused:                 34  (47%)
```

**Nearly half the source is not used** — 34 columns of 72. That number is normal
and it should still be looked at column by column, because the list contains at
least four different things.

**Correctly out of scope.** `employee`, `employee_type` and `payment_type` are
whole tables outside the enrollment process. `home_address`, `postal_code`,
`phone_num`, `email` are operational contact details with no analytical use —
and, being personal data, actively better left out. A warehouse that copies every
column inherits every column's compliance obligations.

**Replaced by something better.** `provn_id` and `cntry_id` are unused because
`dim_student` stores `state` and `country` as *names*. That is a deliberate
denormalisation — worksheet 04's argument — not an oversight.

**Deliberately dropped, and worth recording.** `category_desc`, `course_desc`,
`program_desc` are long free text. `active_flg` on `category` is uniform (all 5
are active, from worksheet 01 question 9), so it carries no information today.
Each is a defensible choice that should be written down, because "we decided not
to" and "we forgot" look the same later.

**The one to question: `gov_code`.** A government course code is exactly the sort
of column an external report will eventually need, and it costs nothing to carry.
`full_time` is similar — a course attribute that someone will want to slice by.

That is the value of this audit. Not to include everything, but to make sure each
exclusion was a decision. The four categories above are four different
conversations, and running this list past whoever owns the source system takes
ten minutes and prevents a rebuild.

### Question 8

Mapping completeness, the other direction. For every column in `TARGET`, classify it as `direct` (copied), `derived` (computed), or `generated` (a surrogate key). Print the counts.
> **NOTE:** this is the column count that tells you how much work the ELT actually is.

In [ ]:
GENERATED = {"program_id", "course_id", "cohort_id", "student_id",
             "date_id", "promotion_id", "enrollment_date_id", "enrollment_id"}
DERIVED = {"program_category", "is_active", "city", "state", "country",
           "full_date", "day_of_week", "day", "week", "month", "quarter",
           "year", "is_weekend", "enrollment_count", "tuition_amount",
           "discount_amount", "net_tuition_amount", "amount_paid_to_date",
           "is_paid_in_full"}
counts = {"direct": 0, "derived": 0, "generated": 0}
for table, cols in TARGET.items():
    LETTER = {"direct": "D", "derived": "E", "generated": "G"}
    kinds = []
    for c in cols:
        k = ("generated" if c in GENERATED
             else "derived" if c in DERIVED else "direct")
        counts[k] += 1
        kinds.append(LETTER[k])
    print("  %-17s %s" % (table, " ".join(kinds)))
print()
print("  D=direct  E=derived  G=generated")
print()
total = sum(counts.values())
for k, v in counts.items():
    print("  %-10s %2d  (%.0f%%)" % (k, v, 100 * v / total))

```
  dim_program       G D D E E
  dim_course        G D D D E
  dim_cohort        G D D D D
  dim_student       G D D D E E E E
  dim_date          G E E E E E E E E
  dim_promotion     G D D E
  fact_enrollment   G G G G G G G E E E E E E

  D=direct  E=derived  G=generated

  direct     14  (29%)
  derived    22  (45%)
  generated  13  (27%)
```

**Only 29% of the target model is copied from the source.** 45% is computed, 27%
is generated.

Which means: **if you thought of this as a data-copying exercise, you would be
wrong about 71% of it.**

Read the rows. `dim_cohort` is `G D D D D` — a surrogate key and four straight
copies, the simplest table in the model. `dim_date` is `G` then eight `E`s: it
has no source table at all, and every column is derived from a date range.
`fact_enrollment` is seven `G`s and six `E`s — **not one direct copy**, because
every key is a surrogate looked up from a dimension and every measure is either a
constant, a summarisation, or a calculation.

The three categories are three different kinds of work, and they fail in three
different ways:

**Direct (14).** Renames and casts. Cheap, and the failure mode is a silent source
schema change.

**Derived (22).** Business logic — `net_tuition_amount = tuition - discount`,
`is_paid_in_full`, `program_category` with its `Unknown` default, the whole date
dimension. This is where the requirements live, where the arguments happen, and
where a wrong answer is invisible. Worksheets 06 and 07 are entirely this column.

**Generated (13).** Surrogate keys, which must be unique, stable across runs, and
correctly resolvable from source keys. Worksheet 08 and 09.

Two practical consequences.

**Estimate from the derived count, not the table count.** "Six dimensions and a
fact table" sounds like a week. Twenty-two derived columns, each needing a rule
agreed with the business and a test, is the actual scope.

**Every `E` needs a written definition.** Slide 41 asks for *"derived measures,
flags, and calculation logic"* in the specification for exactly this reason. A
direct copy is self-documenting; `is_paid_in_full` is not, and two engineers will
implement it differently. Worksheet 07 shows one such definition changing the
headline number.

### Question 9

Slide 34 says staging *"makes extraction results traceable"*. Show what `source_course_id` buys: pick course 214, and print its source row and the target `dim_course` row that would be built from it.
> **NOTE:** the surrogate key is new; the source key is what lets you get back to where a row came from.

In [ ]:
crs = load("course")
dim_course = crs[["course_id", "course_name", "hours", "active_flg"]].copy()
dim_course = dim_course.rename(columns={"course_id": "source_course_id",
                                        "hours": "credit_hours",
                                        "active_flg": "is_active"})
dim_course.insert(0, "course_id", range(1, len(dim_course) + 1))

print("SOURCE row (course.course_id = 214):")
print(crs[crs.course_id == 214][["course_id", "course_name", "hours",
                                 "program_id", "active_flg"]].to_string(index=False))
print()
print("TARGET row in dim_course:")
print(dim_course[dim_course.source_course_id == 214].to_string(index=False))
print()
print("surrogate key differs from source key:",
      int(dim_course.loc[dim_course.source_course_id == 214,
                         "course_id"].iloc[0]) != 214)

```
SOURCE row (course.course_id = 214):
 course_id                    course_name  hours  program_id  active_flg
       214 Observability and SRE Practice     36         106           1

TARGET row in dim_course:
 course_id  source_course_id                    course_name  credit_hours  is_active
        14               214 Observability and SRE Practice            36  1

surrogate key differs from source key: True
```

Two keys on one row. The warehouse calls this course **14**; the source system
calls it **214**; `source_course_id` is the bridge.

Slide 29 draws this on every dimension — `source_program_id`, `source_course_id`,
`source_cohort_id`, `source_student_id`, `source_discount_type_id` — without
explaining it. Four reasons, and the last is the one people learn the hard way.

**Traceability.** A number looks wrong in a report; you trace it to a fact row, to
`dim_course.course_id = 14`, to `source_course_id = 214`, and now you can query
the operational system for course 214 and compare. Without the source key the
trail ends at the warehouse boundary, which is exactly where you need it to
continue. This is slide 34's *"make extraction results traceable"*.

**Independence from source key changes.** Source systems renumber. A migration,
a merger, a system replacement — and the operational `course_id` changes while
the course does not. The surrogate key absorbs that: `dim_course.course_id = 14`
stays 14, historical facts keep pointing at the right course, and only
`source_course_id` is updated.

**Multiple sources.** Acquire a second training provider and its course 214 is a
different course. Two `source_course_id = 214` rows with different
`source_system` values, two distinct surrogate keys, and nothing collides. Source
keys are only unique within their source.

**Slowly changing dimensions.** This is the one that eventually forces the issue.
When a course is renamed and you need history preserved, the Type 2 answer is a
*second row* — same `source_course_id = 214`, new surrogate key, with validity
dates. Facts loaded before the rename point at the old surrogate, facts after at
the new one, and a report of last year shows last year's name. **That is
impossible if the source key is the primary key**, because there can only be one
row per source key.

So the surrogate key is not bureaucracy. It is what makes the dimension able to
have a history at all.

### Question 10

Finally, break slide 38's rule. Build `fact_enrollment` **before** `dim_promotion` exists by looking up promotion keys in an empty dimension: `dim_promotion.set_index("source_discount_type_id").loc[needed]`. **This is supposed to fail.**

In [ ]:
tx = load("transaction")
needed = sorted(int(v) for v in tx.discount_type_id.dropna().unique())
print("promotion keys the fact table needs:", needed)

dim_promotion = pd.DataFrame(columns=["promotion_id", "source_discount_type_id"])
print("rows in dim_promotion at this point:", len(dim_promotion))
print()
print(dim_promotion.set_index("source_discount_type_id").loc[needed])

```
promotion keys the fact table needs: [10, 20, 30, 40, 50, 60]
rows in dim_promotion at this point: 0

KeyError: "None of [Index([10, 20, 30, 40, 50, 60], dtype='int64', name='source_discount_type_id')] are in the [index]"
```

The fact load asks for six promotion keys and the dimension is empty. Slide 38's
rule — *"loading dimensions before facts so fact rows can reference them"* — is
not a style preference; it is a hard dependency, and this is what violating it
costs.

The error is the good outcome. `.loc[]` with a list raises when **none** of the
labels are present, so the load stops at the point of failure with a message
naming the missing keys. What you want to avoid is the same mistake expressed as
a join:

```python
fact.merge(dim_promotion, on="source_discount_type_id", how="left")
```

That does not raise. It returns every fact row with a null `promotion_id`, the
load "succeeds", and `fact_enrollment` arrives with 2,400 rows and a completely
unpopulated foreign key. Every report sliced by promotion returns nothing, and
the pipeline's own row-count check passes — because the row count is right.

Three defences, in order of preference:

**Order the load correctly**, which is question 4's waves and costs nothing.

**Assert after the lookup.** After resolving keys, check that no fact row has a
null dimension key it should not have:

```python
assert fact.promotion_id.notna().sum() == expected, "unresolved promotion keys"
```

Worksheet 10 makes this one of the standard data-quality checks — slide 39's
*"referential integrity check"*.

**Use an `Unknown` member.** Give every dimension a row with key `-1` and label
`Unknown`, and resolve unmatched keys to it rather than to null. Unmatched facts
stay countable, appear in reports as `Unknown`, and someone asks about them —
which is the whole argument from worksheet 01 question 4, applied to keys instead
of labels.

**What this sheet established:**

| | |
|---|---|
| the mapping | 5 columns for `dim_course`, only 2 of them direct copies |
| staging | 2,400 rows, values unchanged — nothing cleaned, on purpose |
| source coverage | 9 of 12 tables used; `payment_type` is out of scope, not irrelevant |
| load order | 3 waves, and wave 2's six dimensions are fully parallel in a star |
| join and filter | 5 joins, row count never moved, grain verified |
| exclusion candidates | 18 + 117 + 7 = **142 by addition, 141 distinct** |
| unused source columns | **34 of 72 — 47%**, in four different categories |
| target column origins | **29% direct, 45% derived, 27% generated** |

Worksheet 06 is Step 3: the business rules that turn those 22 derived columns
from a list into decisions.